<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset MNIST handwritten digits utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (MNIST handwritten digits)

- Utilize os arquivos `*.csv` disponibilizados via Google Classroom
- Use: `as_frame=False`
- Use: `mnist_784`

# Preparação do Notebook

In [2]:
# Garantir a instalação das dependências

%pip install pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [26]:
#Executar os imports

import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.tree import DecisionTreeClassifier

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `MNIST handwritten digits` (arquivos de treino e teste)
Realize a separação do conjunto de treino como treino e validação
Utilize `train_test_split` com controle de aleatoriedade (seed)
Retorne: `X_train`, `X_val`, `y_train`, `y_val`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

***Para garantir reprodutibilidade***

1. Estes arquivos podem também ser encontrados e baixados no link https://www.kaggle.com/datasets/oddrationale/mnist-in-csv
2. Colocar `mnist_train.csv` e `mnist_test.csv` na mesma pasta do notebook
3. Instale as dependências: `pip install pandas scikit-learn`
4. Execute todas as células

In [4]:
#Criação e implementação da função load_data(seed)

def load_data(seed: int):
    # Caminho dos arquivos (modificar quando usado em outra máquina e outro usuário)
    train_path = r"C:\Users\pheli\Downloads\mnist_train.csv"
    test_path = r"C:\Users\pheli\Downloads\mnist_test.csv"
    
    # Carrega os datasets
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # Separa features e labels
    X = train_df.drop("label", axis=1).values
    y = train_df["label"].values
    
    # Divide em treino e validação
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )
    
    return X_train, X_val, y_train, y_val



In [6]:
#Metadados básicos

import pandas as pd

# Carregar os arquivos
train_df = pd.read_csv(r"C:\Users\pheli\Downloads\mnist_train.csv")
test_df = pd.read_csv(r"C:\Users\pheli\Downloads\mnist_test.csv")

# Informações gerais
print("Informações gerais - Treino:")
print(train_df.info())
print("\nInformações gerais - Teste:")
print(test_df.info())

# Primeiras linhas (treino)
print("\nAmostra do treino (primeiras 5 linhas):")
print(train_df.head())

X_train, X_val, y_train, y_val = load_data(seed=42)

print("=== Shapes ===")
print(f"X_train : {X_train.shape}")
print(f"X_val   : {X_val.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_val   : {y_val.shape}")

print("\n=== Tipos ===")
print(f"X dtype : {X_train.dtype}")
print(f"y dtype : {y_train.dtype}")

print("\n=== Intervalo de pixels ===")
print(f"min={X_train.min():.4f}  max={X_train.max():.4f}")

print("\n=== Distribuição de classes (y_train) ===")
unique, counts = np.unique(y_train, return_counts=True)
for cls, cnt in zip(unique, counts):
    print(f"  Dígito {cls}: {cnt} amostras")



Informações gerais - Treino:
<class 'pandas.DataFrame'>
RangeIndex: 60000 entries, 0 to 59999
Columns: 785 entries, label to 28x28
dtypes: int64(785)
memory usage: 359.3 MB
None

Informações gerais - Teste:
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Columns: 785 entries, label to 28x28
dtypes: int64(785)
memory usage: 59.9 MB
None

Amostra do treino (primeiras 5 linhas):
   label  1x1  1x2  1x3  1x4  1x5  1x6  1x7  1x8  1x9  ...  28x19  28x20  \
0      5    0    0    0    0    0    0    0    0    0  ...      0      0   
1      0    0    0    0    0    0    0    0    0    0  ...      0      0   
2      4    0    0    0    0    0    0    0    0    0  ...      0      0   
3      1    0    0    0    0    0    0    0    0    0  ...      0      0   
4      9    0    0    0    0    0    0    0    0    0  ...      0      0   

   28x21  28x22  28x23  28x24  28x25  28x26  28x27  28x28  
0      0      0      0      0      0      0      0      0  
1      0      0      0      

### Deve-se Normalizar?

Podemos notar que todos os valores oscilam entre 0 e 255, o que ao rigor da necessidade de normalização não haveria problemas de escala. Entretanto, ainda assim, podemos considerar a amplitude de 0 a 255 alta, o que poderiam ampliticar-se principalmente nos cálculos de distância Euclidianas, via KNN e quando há o uso de pesos, em especial SVM ou redes neurais. Salvo uso único e exclusivo de árvore de decisão, normalizar os valores é uma solução que traz mais ganhos do que riscos.

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [7]:
#Criação e implementação da função train_random_forest
 
from sklearn.ensemble import RandomForestClassifier

def train_random_forest(X_train, y_train, seed: int):
    """
    Treina um modelo Random Forest com sklearn.
    
    Parâmetros:
        X_train (array): Features de treino
        y_train (array): Labels de treino
        seed (int): Semente para reprodutibilidade
    
    Retorna:
        model (RandomForestClassifier): Modelo treinado
    """
    model = RandomForestClassifier(
        n_estimators=100,      # número de árvores definido para 100
        max_depth=None,        # profundidade definida como ilimitada
        random_state=seed,     # para garantir reprodutibilidade
        n_jobs=-1              # acelera o treino, usando todos os núcleos disponíveis do processador
    )
    
    model.fit(X_train, y_train)
    return model


In [8]:
#Usando e testando

# Treina o modelo
rf_model = train_random_forest(X_train, y_train, seed=42)
print("Modelo treinado:", rf_model)

# Avalia o conjunto de validação
val_accuracy = rf_model.score(X_val, y_val)
print(f"Acurácia na validação: {val_accuracy:.4f}")

Modelo treinado: RandomForestClassifier(n_jobs=-1, random_state=42)
Acurácia na validação: 0.9665


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [9]:
#Criação e implementação da função evaluate

from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    """
    Avalia um modelo treinado em dados de teste.
    
    Parâmetros:
        model: Modelo já treinado (ex: RandomForestClassifier)
        X_test (array): Features do conjunto de teste
        y_test (array): Labels do conjunto de teste
    
    Retorna:
        accuracy (float): Acurácia do modelo
    """
    # Predições
    y_pred = model.predict(X_test)
    
    # Calcular acurácia
    accuracy = accuracy_score(y_test, y_pred)
    
    return accuracy


In [10]:
# O modelo já treinado anteriormente
rf_model = train_random_forest(X_train, y_train, seed=42)

# Avaliar no conjunto de validação
val_acc = evaluate(rf_model, X_val, y_val)
print(f"Acurácia na validação: {val_acc:.4f}")

Acurácia na validação: 0.9665


**O modelo obteve uma acurácia na validação de 0.9665**.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [11]:
#Criação e implementação da função run_pipeline

def run_pipeline(model_type="rf", seed=42):
    X_train, X_val, y_train, y_val = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    else:
        raise ValueError(f"Modelo '{model_type}' não reconhecido.")

    acc = evaluate(model, X_val, y_val)
    print(f"[{model_type.upper()}] seed={seed} | Acurácia: {acc:.4f}")
    return acc


run_pipeline(model_type="rf", seed=42)

[RF] seed=42 | Acurácia: 0.9665


0.9665

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

Em árvores de decisão isoladas (sem random forest), a literatura indica que o overfitting começa a aparecer em profundidadesa partir de 10 a 15. Com `max_depth=None`, a árvore fica livre para cresce até separar perfeitamente cada exemplo do treino, o que leva a praticamente 100% de acurácia no treino.

### Para verificar se houve overfiting:

In [20]:
#Cria nova função a partir da run_pipeline

def run_pipeline_overf(model_type="rf", seed=42):
    X_train, X_val, y_train, y_val = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    else:
        raise ValueError(f"Modelo '{model_type}' não reconhecido.")

    acc_treino = evaluate(model, X_train, y_train)  # <-- avalia no treino
    acc_val    = evaluate(model, X_val, y_val)       # <-- avalia na validação

    print("i   - Se treino ≈ Validação → sem overfitting, o modelo generalizou bem")
    print("ii  - Se treino muito maior que Validação → overfitting, o modelo memorizou o treino")
    print("iii - Se ambos baixos → underfitting, o modelo não aprendeu nem o treino")
    print(f"\n\n[{model_type.upper()}] seed={seed}")
    print(f"  Acurácia no TREINO    : {acc_treino:.4f}")
    print(f"  Acurácia na VALIDAÇÃO : {acc_val:.4f}")
    print(f"  Diferença             : {acc_treino - acc_val:.4f}")

    return acc_val

run_pipeline_overf(model_type="rf", seed=42)

i   - Se treino ≈ Validação → sem overfitting, o modelo generalizou bem
ii  - Se treino muito maior que Validação → overfitting, o modelo memorizou o treino
iii - Se ambos baixos → underfitting, o modelo não aprendeu nem o treino


[RF] seed=42
  Acurácia no TREINO    : 1.0000
  Acurácia na VALIDAÇÃO : 0.9665
  Diferença             : 0.0335


0.9665

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest

## Apresente:
- Acurácia, Precisão, Recall e F1-Score para o modelo

**Solução**:

In [22]:
X_train, X_val, y_train, y_val = load_data(seed=42)
rf_model = train_random_forest(X_train, y_train, seed=42)

y_pred = rf_model.predict(X_val)

acc  = accuracy_score(y_val, y_pred)
prec = precision_score(y_val, y_pred, average="macro")
rec  = recall_score(y_val, y_pred, average="macro")
f1   = f1_score(y_val, y_pred, average="macro")

print("=== Random Forest ===")
print(f"Acurácia : {acc:.4f}")
print(f"Precisão : {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-Score : {f1:.4f}")
print()
print(classification_report(y_val, y_pred))

=== Random Forest ===
Acurácia : 0.9665
Precisão : 0.9664
Recall   : 0.9663
F1-Score : 0.9663

              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1185
           1       0.97      0.98      0.98      1348
           2       0.95      0.97      0.96      1192
           3       0.96      0.95      0.95      1226
           4       0.97      0.97      0.97      1168
           5       0.96      0.96      0.96      1084
           6       0.98      0.98      0.98      1184
           7       0.98      0.96      0.97      1253
           8       0.95      0.95      0.95      1170
           9       0.95      0.95      0.95      1190

    accuracy                           0.97     12000
   macro avg       0.97      0.97      0.97     12000
weighted avg       0.97      0.97      0.97     12000



# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

In [23]:
for seed in [42, 7]:
    run_pipeline(model_type="rf", seed=seed)

[RF] seed=42 | Acurácia: 0.9665
[RF] seed=7 | Acurácia: 0.9689


**Resposta**:

Para seed=42 | Acurácia: 0.9665, para seed=7 | Acurácia: 0.9689.
Houve uma leve mudança nos resultados, aumentando a acurácia em 0.0024 para seed=7. A divisão do treino e validação e a construção das árvores dar-se de forma aleatória, porém, ao fixar o seed a um valor fixo, utilizando-se o random forest, definindo *random_state=seed*, o resultado convergirá sempre para o mesmo valor, garantindo a repodutibilidade do experimento.

# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [ ]:
#Aproveitando a função já criada na observação adicional da questão 4, implementando Random Forest, seed=42:

print("Random Forest:")
run_pipeline_overf(model_type="rf", seed=42)

i   - Se treino ≈ Validação → sem overfitting, o modelo generalizou bem
ii  - Se treino muito maior que Validação → overfitting, o modelo memorizou o treino
iii - Se ambos baixos → underfitting, o modelo não aprendeu nem o treino


[RF] seed=42
  Acurácia no TREINO    : 1.0000
  Acurácia na VALIDAÇÃO : 0.9665
  Diferença             : 0.0335


0.9665

In [ ]:
# Criando uma árvore isolada e sem limite de profundidade
arvore = DecisionTreeClassifier(random_state=42)
arvore.fit(X_train, y_train)

acc_treino_arv = evaluate(arvore, X_train, y_train)
acc_val_arv    = evaluate(arvore, X_val, y_val)

print("=== Árvore de Decisão Isolada ===")
print(f"Acurácia no TREINO    : {acc_treino_arv:.4f}")
print(f"Acurácia na VALIDAÇÃO : {acc_val_arv:.4f}")
print(f"Diferença             : {acc_treino_arv - acc_val_arv:.4f}")

=== Árvore de Decisão ===
Acurácia no TREINO    : 1.0000
Acurácia na VALIDAÇÃO : 0.8701
Diferença             : 0.1299


**Comentários**:

Mantendo-se o seed = 42 (Random Forest), houve 100% de acurácia no treino, o que é forte indicação de ter havido overfiting. 
Ao fazer uma árvore isolada, nota-se ainda 100% de acurácia no treino, mas que vai para 87% no teste, indicando portanto um grande indício de overfiting.
O modelo de árvore isolada causará mais overfiting.


# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`

## Analise:
- O desempenho muda significativamente?

In [28]:
X_train, X_val, y_train, y_val = load_data(seed=42)

for n in [10, 50, 100, 200]: #Gerando 4 variações para o n-estimator
    model = RandomForestClassifier(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    acc = evaluate(model, X_val, y_val)
    print(f"n_estimators={n:>4} | Acurácia: {acc:.4f}")

n_estimators=  10 | Acurácia: 0.9459
n_estimators=  50 | Acurácia: 0.9634
n_estimators= 100 | Acurácia: 0.9665
n_estimators= 200 | Acurácia: 0.9672


**Resposta:**

Sim, há uma mudança de desempenho, aumetando-se a acurácia a medida que se aumenta o valor de n, entretanto, a medida que este valor aumenta exponencialmente, a acurácia tende a convergir, com ganhos marginais cada vez menores, de forma que deve-se avaliar o ganho computacional até se chegar um valor convergente da acurácia.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?

Acurácia não deve ser vista como a única métrica para se avaliar o desempenho e a confiabilidade dos resultados obtidos em um modelo de Machine Learning, em especial o KNN ou árvores de decisão. Há outras métricas que devem ser usadas para auxiliar na avaliação da qualidade e confiabilidade do modelo: precisão, recall, F1-score. 

A acurácia mede apenas a proporção geral de acertos (Verdadeiros Positivos e Verdadeiros Negativos) em relação ao total de previsões. Para alguns tipos de análise, o mais importante está em detectar os "poucos importantes", que seriam resultados que se ocorressem, poderiam levar a danos catastróficos, e nisto, a acurácia não é a melhor métrica a se levar em consideração.

2. Como você garante que o resultado não ocorreu por acaso?

Deixar o **random_state** com um valor fixo para o seed, tende a levar a reprodutibilidade dos valores dos testes, porém isto só seria melhor evidenciado com números maiores, podendo executar o pipeline com múltiplos valores de seeds, ou por valicação cruzada (cross-validation), que avalia o modelo em diferentes subconjuntos dos dados e calcula média e desvio padrão da acurácia.

3. Cite dois possíveis problemas metodológicos neste experimento.

Faltou uma introdução, o que não especificou o que se estava procurando, focando-se mais em aplicar funções. Sem um objetivo claro, a interpretação dos resultados e do que deve ser melhorado ou mantido ficou mais dispersa e com maior limitação a analisar os retornos dos códigos. Não foi feita uma busca de hiperparâmetros de forma sistemática, para explorar mais potencialidades dos modelos.


4. O pipeline implementado é confiável? Justifique.

Ele garantiu reprodutibilidade via seed e separação correta dos dados. O ideal para se ter mais confiabilidade, é deixar o conjunto de teste separado do desenvolvimento (para evitar possível "dataleaking") fazendo-se um fluxo dos dados estruturado em blocos isolados durante a extração de métricas.